<a href="https://colab.research.google.com/github/AHHHHHH0-0/172b/blob/cnn/cnn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import os
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from tqdm import tqdm

In [12]:
# ==========================================
# 1. Data Loading & Preprocessing
# ==========================================
DATASET_PATH = "./172b/fonts_dataset"

images = []
labels = []

# Load image files and extract labels from filenames
filenames = sorted(os.listdir(DATASET_PATH))
for filename in filenames:
    # Skip macOS metadata files (AppleDouble files) and ensure it's a PNG
    if filename.startswith("._") or not filename.endswith(".png"):
        continue

    # Assuming filename format
    parts = filename.replace(".png", "").split("_")
    font_label = parts[1]

    img_path = os.path.join(DATASET_PATH, filename)
    img = Image.open(img_path).convert("L")  # Convert to grayscale
    img_array = np.array(img)

    images.append(img_array)
    labels.append(font_label)

X = np.array(images)
y = np.array(labels)

print(f"Total loaded images: {len(X)}")

# Encode string labels to integer indices
le = LabelEncoder()
y_encoded = le.fit_transform(y)
num_classes = len(le.classes_)

# Split dataset into training and validation sets (80:20)
X_train, X_val, y_train, y_val = train_test_split(
    X, y_encoded, test_size=0.2, stratify=y_encoded, random_state=42
)
print(f"Train samples: {len(X_train)}, Validation samples: {len(X_val)}")

Total loaded images: 25000
Train samples: 20000, Validation samples: 5000


In [11]:
# Replace 'your_archive_name.tar' with the actual path to your .tar file.
# Replace 'destination_folder' with the desired extraction directory, or omit '-C destination_folder' to extract in the current directory.
!tar -xf /content/drive/MyDrive/fonts_dataset.tar -C /content/

Streaming output truncated to the last 5000 lines.
tar: Ignoring unknown extended header keyword 'LIBARCHIVE.xattr.com.apple.provenance'
tar: Ignoring unknown extended header keyword 'LIBARCHIVE.xattr.com.apple.provenance'
tar: Ignoring unknown extended header keyword 'LIBARCHIVE.xattr.com.apple.provenance'
tar: Ignoring unknown extended header keyword 'LIBARCHIVE.xattr.com.apple.provenance'
tar: Ignoring unknown extended header keyword 'LIBARCHIVE.xattr.com.apple.provenance'
tar: Ignoring unknown extended header keyword 'LIBARCHIVE.xattr.com.apple.provenance'
tar: Ignoring unknown extended header keyword 'LIBARCHIVE.xattr.com.apple.provenance'
tar: Ignoring unknown extended header keyword 'LIBARCHIVE.xattr.com.apple.provenance'
tar: Ignoring unknown extended header keyword 'LIBARCHIVE.xattr.com.apple.provenance'
tar: Ignoring unknown extended header keyword 'LIBARCHIVE.xattr.com.apple.provenance'
tar: Ignoring unknown extended header keyword 'LIBARCHIVE.xattr.com.apple.provenance'
tar

In [13]:
# ==========================================
# 2. Custom Dataset & Data Augmentation
# ==========================================
class AddGaussianNoise(object):
    """Custom transform to add Gaussian noise to images."""
    def __init__(self, mean=0., std=0.05):
        self.mean = mean
        self.std = std

    def __call__(self, tensor):
        return tensor + torch.randn_like(tensor) * self.std + self.mean

    def __repr__(self):
        return f'{self.__class__.__name__}(mean={self.mean}, std={self.std})'


class FontDataset(Dataset):
    """Custom PyTorch Dataset for Font images."""
    def __init__(self, X, y, transform=None):
        self.X = X
        self.y = y
        self.transform = transform

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        image = self.X[idx]  # Shape: (H, W)
        label = self.y[idx]

        if self.transform:
            image = self.transform(image)
        else:
            # Default scaling if no transform is provided
            image = torch.tensor(image, dtype=torch.float32).unsqueeze(0) / 255.0

        return image, torch.tensor(label, dtype=torch.long)


# Define transformation pipelines for train and validation
train_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.RandomRotation(5),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
    transforms.ToTensor(),
    AddGaussianNoise(mean=0., std=0.05),
    transforms.Normalize((0.5,), (0.5,))
])

val_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# Create DataLoaders
batch_size = 64
train_dataset = FontDataset(X_train, y_train, transform=train_transform)
val_dataset = FontDataset(X_val, y_val, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)



In [14]:
# ==========================================
# 3. Model Architecture (CNN)
# ==========================================
class FontCNN(nn.Module):
    """CNN architecture for Font Classification."""
    def __init__(self, num_classes):
        super(FontCNN, self).__init__()

        # Feature extraction layers
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            # Global Average Pooling ensures fixed output size regardless of input spatial dimensions
            nn.AdaptiveAvgPool2d((1, 1))
        )

        # Classification head
        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1) # Flatten (B, C, 1, 1) to (B, C)
        x = self.classifier(x)
        return x



In [15]:
# ==========================================
# 4. Training Setup & Loop
# ==========================================
# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nTraining on device: {device}\n")

# Initialize model, loss function, and optimizer
model = FontCNN(num_classes=num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Learning Rate Scheduler: Reduces LR when validation loss plateaus
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

num_epochs = 15
best_val_loss = float('inf')

# Main training loop
for epoch in range(num_epochs):

    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    # Progress bar for training
    train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Train]")

    for images, labels in train_bar:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        # Display current loss on the progress bar
        train_bar.set_postfix(loss=f"{loss.item():.4f}")

    epoch_train_loss = running_loss / total
    epoch_train_acc = (correct / total) * 100


    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        val_bar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Val]")
        for val_images, val_labels in val_bar:
            val_images, val_labels = val_images.to(device), val_labels.to(device)

            val_outputs = model(val_images)
            v_loss = criterion(val_outputs, val_labels)

            val_loss += v_loss.item() * val_images.size(0)
            _, val_predicted = torch.max(val_outputs, 1)
            val_total += val_labels.size(0)
            val_correct += (val_predicted == val_labels).sum().item()

    epoch_val_loss = val_loss / val_total
    epoch_val_acc = (val_correct / val_total) * 100

    # Step the learning rate scheduler
    scheduler.step(epoch_val_loss)


    print(f"\n--- Epoch {epoch+1} Summary ---")
    print(f"Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc:.2f}%")
    print(f"Val Loss:   {epoch_val_loss:.4f} | Val Acc:   {epoch_val_acc:.2f}%")


    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        torch.save(model.state_dict(), "best_font_cnn.pth")
        print(f">>> Best model saved to 'best_font_cnn.pth' (Loss: {best_val_loss:.4f})")

    print("-" * 50)

print("\nTraining completed successfully!")


Training on device: cuda



Epoch 1/15 [Val]: 100%|██████████| 79/79 [00:04<00:00, 17.42it/s]



--- Epoch 1 Summary ---
Train Loss: 1.2516 | Train Acc: 47.98%
Val Loss:   4.4527 | Val Acc:   22.34%
>>> Best model saved to 'best_font_cnn.pth' (Loss: 4.4527)
--------------------------------------------------


Epoch 2/15 [Val]: 100%|██████████| 79/79 [00:04<00:00, 18.09it/s]



--- Epoch 2 Summary ---
Train Loss: 0.5040 | Train Acc: 85.00%
Val Loss:   8.6295 | Val Acc:   34.76%
--------------------------------------------------


...


--- Epoch 13 Summary ---
Train Loss: 0.0986 | Train Acc: 96.77%
Val Loss:   2.6240 | Val Acc:   65.14%
--------------------------------------------------


Epoch 14/15 [Val]: 100%|██████████| 79/79 [00:04<00:00, 17.28it/s]



--- Epoch 14 Summary ---
Train Loss: 0.0859 | Train Acc: 97.28%
Val Loss:   3.7044 | Val Acc:   64.04%
--------------------------------------------------


Epoch 15/15 [Val]: 100%|██████████| 79/79 [00:04<00:00, 17.30it/s]


--- Epoch 15 Summary ---
Train Loss: 0.0824 | Train Acc: 97.20%
Val Loss:   4.6307 | Val Acc:   62.66%
--------------------------------------------------

Training completed successfully!
